# Day 40：猫狗数据准备与可复现划分

目标：准备不交叉的训练、验证、测试清单，而不是一次把全部图像转为巨大 pickle。请先按[数据说明](../datasets/readme.md)下载并解压 PetImages/Dog 和 PetImages/Cat；数据不随仓库提供。

运行前请阅读[环境与运行说明](../docs/setup.md)。本课 `.py` 是教学源文件，配套 Markdown 和 Notebook 自动同步。图形保存到 `outputs/`，设置 `COURSE_SHOW_PLOTS=1` 可显示窗口。


[Python 源文件](Day%2040.py) · [Notebook](Day%2040.ipynb) · [完整课程目录](../docs/curriculum.md)


In [ ]:
from pathlib import Path
import sys

# 脚本从文件位置定位仓库；Notebook 从当前工作目录向上查找。
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for candidate in (base, *base.parents):
    if (candidate / "Code" / "course_utils.py").is_file():
        code_dir = str(candidate / "Code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        break
else:
    raise FileNotFoundError("找不到课程仓库，请从仓库根目录或 Code 目录启动 Notebook。")
from course_utils import DATA, OUTPUT, finish_plot


## 检查、去重和分层划分

共享函数按固定随机顺序读取候选文件，达到每类样本上限后停止；记录的是已检查部分的坏图，不是全库审计。它检查可解码性，按文件字节 SHA-256 排除完全重复，再在每个类别内按固定种子分为 60%/20%/20%。相似但不完全相同的照片仍可能有关联，如有主体信息应采用分组划分。快速验证每类最多 20 张；正常默认每类最多 500 张，让 CPU 入门训练更可控。


In [ ]:
import os
from collections import Counter
from deep_utils import configure, pet_root, prepare_pets, save_manifest
configure()
limit = 20 if os.environ.get("COURSE_SMOKE") == "1" else int(os.environ.get("COURSE_PET_LIMIT", "500"))
if limit < 10:
    raise ValueError("COURSE_PET_LIMIT must be at least 10")
manifest = prepare_pets(pet_root(), limit_per_class=limit)
manifest_path = save_manifest(manifest)
print("Split/class counts:", Counter((r["split"], r["label"]) for r in manifest["records"]))
print("Bad/duplicate examples:", manifest["rejected"][:5])


## 数据清单与批次读取的关系

`prepare_pets` 返回字典；每条 records 包含相对路径 path、标签 label、文件哈希 sha256、所属 split。
它先按类切分，保证每部分都有猫和狗。整数取整时，训练和验证数量向下取整，余下样本归测试集。
`pet_dataset` 先构造小整数索引，训练时完整打乱索引，再解码对应图像，避免缓存全部图片。
每轮重新打乱，但从同一种子新建 Dataset 时可复现顺序。验证和测试始终保留清单顺序。
共享实现位于 [deep_utils.py](deep_utils.py)，是数据读取的辅助函数，不执行模型训练。

## 读取一个批次

每次只解码所需图像；采用 RGB 三通道，缩放为 64×64×3，float32 范围 [0,1]。标签 0=Dog、1=Cat。清单保存到 outputs，可供 Day 41/42 使用；不要在训练后重新划分再比较结果。


In [ ]:
import matplotlib.pyplot as plt
from deep_utils import pet_dataset
batch_images, batch_labels = next(iter(pet_dataset(manifest, "train")))
print("Batch:", batch_images.shape, batch_images.dtype, "labels:", batch_labels.numpy())
fig, ax = plt.subplots()
ax.imshow(batch_images[0].numpy())
ax.set_title(manifest["class_names"][int(batch_labels[0])])
ax.axis("off")
finish_plot("day40_sample")


## 练习与检查

确认每个 split 都有猫和狗，且哈希集合互不相交。检查拒绝记录；随机打乱不保证类别交替。增大样本上限时记录清单和开销，不覆盖已用于报告结果的划分。
